# Visualization Module Examples

This notebook demonstrates the `glassboxllms.visualization` module, which provides:
- **Static plots** (matplotlib/seaborn) for publication-ready figures
- **Interactive plots** (plotly) for exploration and HTML dashboards

In [ ]:
import numpy as np
np.random.seed(42)

## 1. Attention Heatmap

In [ ]:
from glassboxllms.visualization import plot_attention_heatmap

# Simulate an attention matrix (8 tokens)
tokens = ["The", "cat", "sat", "on", "the", "mat", ".", "<eos>"]
attention = np.random.dirichlet(np.ones(8), size=8)  # rows sum to 1

fig = plot_attention_heatmap(attention, tokens, layer=5, head=3)
fig

## 2. Logit Lens

In [ ]:
from glassboxllms.visualization import plot_logit_lens

# Simulate correct-token probability increasing through layers
n_layers = 12
seq_len = len(tokens)
logit_lens_data = np.zeros((n_layers, seq_len))
for layer in range(n_layers):
    logit_lens_data[layer] = np.clip(
        0.1 + (layer / n_layers) * 0.8 + np.random.randn(seq_len) * 0.05,
        0, 1,
    )

fig = plot_logit_lens(logit_lens_data, tokens)
fig

## 3. SAE Training Curves

In [ ]:
from glassboxllms.visualization import plot_sae_training_curves

# Simulate training history (as produced by SAETrainer)
n_steps = 200
training_history = {
    "loss": list(np.exp(-np.linspace(0, 3, n_steps)) + np.random.randn(n_steps) * 0.01),
    "mse_loss": list(np.exp(-np.linspace(0, 3, n_steps)) * 0.8 + np.random.randn(n_steps) * 0.005),
    "explained_variance": list(1 - np.exp(-np.linspace(0, 3, n_steps))),
    "mean_l0": list(np.linspace(30, 128, n_steps) + np.random.randn(n_steps) * 2),
    "dead_feature_count": list(np.maximum(0, 500 - np.linspace(0, 490, n_steps) + np.random.randn(n_steps) * 10).astype(int)),
}

fig = plot_sae_training_curves(training_history)
fig

## 4. SAE Sparsity Distribution

In [ ]:
from glassboxllms.visualization import plot_sae_sparsity

# Simulate per-feature activation frequencies (most features are sparse)
feature_sparsities = np.random.exponential(0.05, size=4096)
feature_sparsities = np.clip(feature_sparsities, 0, 1)

# Simulate per-sample reconstruction MSE
reconstruction_mse = np.random.exponential(0.02, size=1000)

fig = plot_sae_sparsity(feature_sparsities, reconstruction_mse=reconstruction_mse)
fig

## 5. Probe Accuracy by Layer

In [ ]:
from glassboxllms.visualization import plot_probe_accuracy

# Simulate probe results across layers
probe_results = {}
for i in range(12):
    acc = 0.5 + 0.4 * (i / 11) + np.random.randn() * 0.02
    probe_results[f"layer_{i}"] = {
        "accuracy": np.clip(acc, 0, 1),
        "f1": np.clip(acc - 0.03, 0, 1),
    }

fig = plot_probe_accuracy(probe_results)
fig

## 6. Circuit Graph Rendering

In [ ]:
from glassboxllms.analysis.circuits import CircuitGraph
from glassboxllms.visualization import plot_circuit_graph

# Build a small IOI circuit
graph = CircuitGraph(model="gpt2-small", name="IOI Circuit")

# Embedding
graph.add_node("embed", node_type="embedding", layer=0)

# Attention heads
graph.add_node("attn.1.head.0", node_type="attention_head", layer=1, index=0)
graph.add_node("attn.3.head.2", node_type="attention_head", layer=3, index=2)
graph.add_node("attn.5.head.1", node_type="attention_head", layer=5, index=1)

# MLP neurons
graph.add_node("mlp.2.neuron.42", node_type="neuron", layer=2, index=42)
graph.add_node("mlp.4.neuron.7", node_type="neuron", layer=4, index=7)

# Output
graph.add_node("unembed", node_type="unembedding", layer=6)

# Edges
graph.add_edge("embed", "attn.1.head.0", weight=0.6)
graph.add_edge("attn.1.head.0", "mlp.2.neuron.42", weight=0.8)
graph.add_edge("mlp.2.neuron.42", "attn.3.head.2", weight=0.5)
graph.add_edge("attn.3.head.2", "mlp.4.neuron.7", weight=0.7)
graph.add_edge("mlp.4.neuron.7", "attn.5.head.1", weight=0.9)
graph.add_edge("attn.5.head.1", "unembed", weight=0.85)

fig = plot_circuit_graph(graph)
fig

## 7. Steering Effect Comparison

In [ ]:
from glassboxllms.visualization import plot_steering_effects

effects = {
    "baseline": {"sentiment_score": 0.05, "perplexity": 12.3},
    "+love (1x)": {"sentiment_score": 0.45, "perplexity": 14.1},
    "+love (3x)": {"sentiment_score": 0.82, "perplexity": 18.5},
    "+anger (1x)": {"sentiment_score": -0.38, "perplexity": 15.2},
    "+anger (3x)": {"sentiment_score": -0.71, "perplexity": 22.8},
}

fig = plot_steering_effects(effects, metric="sentiment_score")
fig

---
## Interactive Plots (Plotly)

### 8. SAE Feature Browser

In [ ]:
from glassboxllms.visualization import feature_browser

# Simulate feature activations: 100 features x 20 token positions
# Most features are sparse (many zeros)
activations = np.random.exponential(0.1, size=(100, 20))
activations[activations < 0.15] = 0  # enforce sparsity

feature_labels = [f"Feature {i}" for i in range(100)]
token_labels = ["The", "quick", "brown", "fox", "jumps", "over", "the",
                "lazy", "dog", ".", "<eos>", "It", "was", "a", "sunny",
                "day", "in", "the", "park", "."]

fig = feature_browser(
    activations,
    feature_labels=feature_labels,
    token_labels=token_labels,
    top_k=15,
)
fig.show()

### 9. Interactive Circuit Graph Explorer

In [ ]:
from glassboxllms.visualization import interactive_circuit_explorer

# Re-use the circuit graph from above
fig = interactive_circuit_explorer(graph, title="IOI Circuit Explorer")
fig.show()

### 10. Embedding Space 3D Scatter

In [ ]:
from glassboxllms.visualization import embedding_scatter_3d

# Simulate embeddings from 3 clusters
n_per_cluster = 100
dim = 768

cluster_centers = np.random.randn(3, dim) * 5
embeddings = np.vstack([
    cluster_centers[i] + np.random.randn(n_per_cluster, dim) * 0.5
    for i in range(3)
])
labels = ["Cluster A"] * n_per_cluster + ["Cluster B"] * n_per_cluster + ["Cluster C"] * n_per_cluster

fig = embedding_scatter_3d(embeddings, labels=labels, method="pca", title="Token Embeddings")
fig.show()

### Saving to HTML

All interactive plots support `save_path` to export standalone HTML files:

In [ ]:
# Example: save interactive circuit to HTML
# fig = interactive_circuit_explorer(graph, save_path="circuit_explorer.html")
# fig = embedding_scatter_3d(embeddings, labels=labels, method="pca", save_path="embeddings.html")